In [ ]:
import re
import os
import matplotlib.pyplot as plt
import numpy as np

def parse_log_to_dicts(file_path):
    """
    Parses a log file to extract execution times into two dictionaries:
    one for PG (default) and one for Model (BAO/GNTO).
    
    Returns:
        pg_dict: {query: [time1, time2, ...]}
        model_dict: {query: [time1, time2, ...]}
    """
    pg_dict = {}
    model_dict = {}
    
    # Regex patterns
    # Matches: BAO 22a_job.sql 1.05 or GNTO ...
    model_pattern = re.compile(r"(?:BAO|GNTO)\s+(\S+)\s+([\d\.]+)")
    # Matches: x x 22a_job.sql 0.44 PG
    pg_pattern = re.compile(r"x\s+x\s+(\S+)\s+([\d\.]+)\s+PG")
    
    print(f"Parsing {file_path} for dicts...")
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                
                # Check for Model pattern
                model_match = model_pattern.search(line)
                if model_match:
                    query = model_match.group(1)
                    time_val = float(model_match.group(2))
                    if query not in model_dict:
                        model_dict[query] = []
                    model_dict[query].append(time_val)
                    continue
                
                # Check for PG pattern
                pg_match = pg_pattern.search(line)
                if pg_match:
                    query = pg_match.group(1)
                    time_val = float(pg_match.group(2))
                    if query not in pg_dict:
                        pg_dict[query] = []
                    pg_dict[query].append(time_val)
                    
    except FileNotFoundError:
        print(f"Error: File {file_path} not found.")
        return {}, {}
        
    return pg_dict, model_dict

pg_times, gnto_model_times = parse_log_to_dicts("/home/AiChaosN/Project/Phd/project/LIMAOLifeLongRLDB/gnto_ex/tmp/gnto_log")
_, bao_model_times = parse_log_to_dicts("/home/AiChaosN/Project/Phd/project/LIMAOLifeLongRLDB/gnto_ex/tmp/bao_log")

Parsing /home/AiChaosN/Project/Phd/project/LIMAOLifeLongRLDB/gnto_ex/tmp/gnto_log for dicts...
Parsing /home/AiChaosN/Project/Phd/project/LIMAOLifeLongRLDB/gnto_ex/tmp/bao_log for dicts...


In [4]:

# 打印某个查询的所有运行时间
if '22a_job.sql' in pg_times:
    print("PG Times:", pg_times['22a_job.sql'])
if '22a_job.sql' in gnto_model_times:
    print("GNTO Model Times:", gnto_model_times['22a_job.sql'])
if '22a_job.sql' in bao_model_times:
    print("BAO Model Times:", bao_model_times['22a_job.sql'])

PG Times: [18.802397966384888]
GNTO Model Times: [32.0, 1.0423643589019775, 1.0542631149291992]
BAO Model Times: [4.1675097942352295]


In [15]:

def plot_multi_dict_comparison(dict_list, label_list, title, output_filename, output_dir):
    """
    Plots a grouped bar chart comparing the last execution time for each query 
    across multiple dictionaries.
    
    Args:
        dict_list: List of dictionaries {query: [time1, time2, ...]}
        label_list: List of labels corresponding to each dictionary (e.g., ['PG', 'BAO', 'GNTO'])
        title: Plot title
        output_filename: Filename to save
        output_dir: Directory to save
    """
    if not dict_list or not label_list or len(dict_list) != len(label_list):
        print("Error: Invalid input lists for comparison.")
        return

    # 1. Collect all unique queries (keys)
    all_queries = set()
    for d in dict_list:
        all_queries.update(d.keys())
    
    # Sort queries naturally or alphabetically
    queries = sorted(list(all_queries))
    
    if not queries:
        print(f"No queries found to plot for {title}.")
        return

    # 2. Extract data for plotting (last value of each list)
    plot_data = [] # List of lists, one per category
    for d in dict_list:
        category_values = []
        for q in queries:
            if q in d and d[q]:
                # Take the last value as requested
                category_values.append(d[q][-1])
            else:
                # Missing data for this query in this category
                category_values.append(0) 
        plot_data.append(category_values)

    # 如果存在0或者32.0，对应的query都不显示（需要保持各list长度与queries一致）
    # 筛选要保留的queries（所有category都不为0且不为32.0的）
    valid_idx = [
        i for i in range(len(queries))
        if all(plot_data[j][i] != 0 and plot_data[j][i] != 32.0 and plot_data[j][i] < 7 and plot_data[j][i] > 0.3 for j in range(len(plot_data)))
    ]
    queries = [queries[i] for i in valid_idx]
    plot_data = [[y[i] for i in valid_idx] for y in plot_data]


    # 3. Plotting
    n_queries = len(queries)
    n_categories = len(dict_list)
    
    # Dynamic height
    fig_height = max(6, n_queries * 0.4)
    plt.figure(figsize=(12, fig_height))
    
    y = np.arange(n_queries)
    total_height = 0.8 # Total height available for a group of bars
    bar_height = total_height / n_categories
    
    # Simple color cycle
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    
    for i in range(n_categories):
        # Calculate y-offset to group bars around the tick
        # i=0 starts at top (since y axis inverted later), but mathematically:
        # We want to center the group at y.
        # Start offset: - (total_height / 2) + (bar_height / 2)
        # Shift per item: + i * bar_height
        
        # Center of the group is 'y'.
        # Top-most bar (visually) corresponds to lowest y-value modification if not inverted yet...
        # Let's keep it simple: 
        # offset = (i - n_categories/2 + 0.5) * bar_height
        offset = (i - (n_categories - 1) / 2) * bar_height
        
        c = colors[i % len(colors)]
        plt.barh(y + offset, plot_data[i], height=bar_height, label=label_list[i], color=c, alpha=0.8)

    plt.yticks(y, queries)
    plt.xlabel('Execution Time (s)')
    plt.title(title)
    plt.legend()
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    
    # Invert y axis so top queries are at top
    plt.gca().invert_yaxis()
    
    save_path = os.path.join(output_dir, output_filename)
    plt.tight_layout()
    plt.savefig(save_path)
    print(f"Saved {output_filename}")
    plt.close()


# 准备数据
dicts_to_plot = [pg_times, bao_model_times, gnto_model_times]
labels = ["PG Default", "BAO Model", "GNTO Model"]

# 画图
plot_multi_dict_comparison(
    dicts_to_plot, 
    labels, 
    "Model Performance Comparison (Last Execution)", 
    "multi_model_comparison.png", 
    "."
)

Saved multi_model_comparison.png
